# Day 9 — Loss Functions for Imbalanced Segmentation

Day 8 established the diagnosis: on LandCover.ai, building occupies **0.23%** of pixels
and woodland **60.89%** — a 265:1 ratio. Plain cross-entropy averages over pixels, so a
class at 0.23% barely affects the loss, and the model settles on predicting the
majority classes.

This notebook tests three fixes, each with a different mechanism, plus one arm added
after the first results exposed a flaw in the experimental design.

## Result

| loss | pixAcc | mIoU | building IoU | road IoU |
|---|---|---|---|---|
| cross-entropy | **0.8677** | 0.4423 | **0.0000** | 0.0136 |
| weighted CE | 0.8229 | 0.4978 | 0.2509 | 0.1873 |
| soft Dice | 0.8317 | 0.4987 | 0.2283 | 0.2265 |
| weighted CE + Dice | 0.8014 | 0.4572 | 0.1857 | 0.1081 |
| **plain CE + Dice** | 0.8566 | **0.5192** | 0.2257 | 0.2014 |

**The arm with the highest pixel accuracy is the only one that never predicts a
building.** Cross-entropy scores 0.8677 — best of all five — with a building IoU of
exactly zero. Judged on accuracy, it wins; judged on what it can actually segment, it
is the worst model here.

## Three findings

1. **Plain cross-entropy fails completely on a 0.23% class.** Building IoU 0.0000, road
   recall 0.015 — 98.5% of road pixels missed.
2. **Weighted CE and Dice score identically but work differently.** mIoU 0.4978 vs
   0.4987, indistinguishable. But on road, weighted CE gets recall 0.471 / precision
   0.237, while Dice gets recall 0.319 / precision 0.438 — opposite trade-offs reaching
   the same IoU. A single IoU number hides this entirely.
3. **The predicted winner lost, because of a design flaw — and fixing it produced the
   actual winner.** `CEPlusDice` was built with *weighted* CE, so both terms pushed
   toward over-predicting rare classes. Its building precision of 0.202 against recall
   0.692 is the signature. Re-running with unweighted CE raised precision to 0.342
   (+69%) and produced the best mIoU of the five.

The design flaw was visible only because precision and recall were logged alongside
IoU. That instrumentation choice is the reason this notebook has a fourth section.

**Environment:** run locally on Apple Silicon (MPS), not Colab. Device selection,
DataLoader settings and MPS seeding differ from the CUDA notebooks accordingly.

## 0. Setup

Three MPS-specific points, none of which apply on CUDA:

- **`PYTORCH_ENABLE_MPS_FALLBACK=1`** must be set *before* torch is imported. Without
  it, an operator MPS does not implement raises an error instead of falling back to CPU.
- **`pin_memory=False`** — pinned host memory is a CUDA concept.
- **`num_workers=0`** — macOS spawns worker processes rather than forking, which
  interacts badly with Jupyter. Raise it later only if the speed test says I/O is the
  bottleneck.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"     # must precede `import torch`

import time, copy, glob, zipfile
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
print("device:", device, "| torch:", torch.__version__)

def seed_all(s=42):
    torch.manual_seed(s)
    if device == "mps":
        torch.mps.manual_seed(s)                    # manual_seed does not cover MPS
    elif device == "cuda":
        torch.cuda.manual_seed_all(s)

def sync():
    """MPS and CUDA are asynchronous -- timing without this measures how long it
    took to *queue* the work, not to do it."""
    if device == "mps":
        torch.mps.synchronize()
    elif device == "cuda":
        torch.cuda.synchronize()

## 1. Data

LandCover.ai, pre-tiled 512×512 chips with the official train/val/test split (cut by
orthophoto, not by random chip — so no spatial leakage between splits).

**License: CC-BY-NC-SA-4.0**, non-commercial, share-alike. Source:
<https://landcover.ai.linuxpolska.com/>. Cite Boguszewski et al., CVPRW 2021.

HuggingFace caches the archive under `~/.cache/huggingface`, so unlike Colab this
downloads once and persists.

In [ ]:
from huggingface_hub import hf_hub_download

zp = hf_hub_download(
    repo_id="MortenTabaka/LandCover-Aerial-Imagery-for-semantic-segmentation",
    filename="landcover_processed_for_training.zip",
    repo_type="dataset",
)
DEST = os.path.expanduser("~/dl-local/landcover")
if not os.path.exists(f"{DEST}/processed/train"):
    os.makedirs(DEST, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(DEST)

for split in ("train", "val", "test"):
    n_i = len(glob.glob(f"{DEST}/processed/{split}/images/**/*.jpg", recursive=True))
    n_m = len(glob.glob(f"{DEST}/processed/{split}/masks/**/*.png",  recursive=True))
    print(f"{split:6s} images {n_i:6d}  masks {n_m:6d}")

### Dataset

Two failure modes here are **silent** rather than loud:

- **Masks must be resized with `NEAREST`.** Bilinear interpolation on class indices
  invents values that were never labelled — between building (1) and water (3) it
  produces 2, i.e. spurious woodland.
- **Image/mask pairing must be asserted.** If the two `glob` calls order differently,
  the model trains image A against mask B. Loss still falls (it learns the class prior),
  IoU is inexplicably low, nothing raises.

In [ ]:
SIZE = 256
N_CLASSES = 5
CLASS_NAMES = ["background", "building", "woodland", "water", "road"]

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def _stem(p):
    return os.path.splitext(os.path.basename(p))[0]

class LandCoverDS(Dataset):
    def __init__(self, split, size=SIZE, limit=None):
        root = f"{DEST}/processed/{split}"
        self.imgs = sorted(glob.glob(f"{root}/images/**/*.jpg", recursive=True))
        self.msks = sorted(glob.glob(f"{root}/masks/**/*.png",  recursive=True))
        assert len(self.imgs) == len(self.msks) > 0, f"{split}: {len(self.imgs)}/{len(self.msks)}"
        bad = [(a, b) for a, b in zip(self.imgs, self.msks) if _stem(a) != _stem(b)]
        assert not bad, f"{len(bad)} mismatched pairs, first: {bad[0]}"
        if limit:
            self.imgs, self.msks = self.imgs[:limit], self.msks[:limit]
        self.size = size

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        img = Image.open(self.imgs[i]).convert("RGB").resize((self.size, self.size), Image.BILINEAR)
        msk = Image.open(self.msks[i]).resize((self.size, self.size), Image.NEAREST)
        x = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        x = (x - MEAN) / STD
        m = np.array(msk)
        m = m[:, :, 0] if m.ndim == 3 else m
        return x, torch.from_numpy(m.astype(np.int64))

BATCH = 16

# Reduced training set. Measured 4.7 min/epoch on the full 7,470 tiles (M5, MPS),
# which is ~2.5 h for four arms. Every arm is affected identically, so the
# between-loss comparison holds; absolute IoU will be lower than Day 8's and is
# NOT comparable to it.
TRAIN_LIMIT, VAL_LIMIT = 2500, 800

train_ds = LandCoverDS("train", limit=TRAIN_LIMIT)
val_ds   = LandCoverDS("val",   limit=VAL_LIMIT)
test_ds  = LandCoverDS("test")

# pin_memory / num_workers tuned for macOS -- see notes above
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=False)
val_dl   = DataLoader(val_ds,   batch_size=32,    shuffle=False, num_workers=0, pin_memory=False)
test_dl  = DataLoader(test_ds,  batch_size=32,    shuffle=False, num_workers=0, pin_memory=False)

print(len(train_ds), len(val_ds), len(test_ds), "|", len(train_dl), "steps/epoch")
xb, yb = next(iter(train_dl))
print("image:", tuple(xb.shape), f"[{xb.min():.2f}, {xb.max():.2f}]")
print("mask :", tuple(yb.shape), "unique:", torch.unique(yb).tolist())

## 2. Model

Identical to Day 8's U-Net (7,763,173 parameters with skips). Repeated here so this
notebook stands alone.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, n_classes=N_CLASSES, base=32, use_skip=True):
        super().__init__()
        self.use_skip = use_skip
        b = base
        self.pool = nn.MaxPool2d(2)
        self.enc1 = DoubleConv(in_ch, b)
        self.enc2 = DoubleConv(b,   b*2)
        self.enc3 = DoubleConv(b*2, b*4)
        self.enc4 = DoubleConv(b*4, b*8)
        self.bottleneck = DoubleConv(b*8, b*16)
        s = (lambda c: c if use_skip else 0)
        self.up4  = nn.ConvTranspose2d(b*16, b*8, 2, stride=2)
        self.dec4 = DoubleConv(b*8 + s(b*8), b*8)
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = DoubleConv(b*4 + s(b*4), b*4)
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = DoubleConv(b*2 + s(b*2), b*2)
        self.up1  = nn.ConvTranspose2d(b*2, b, 2, stride=2)
        self.dec1 = DoubleConv(b + s(b), b)
        self.head = nn.Conv2d(b, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        bn = self.bottleneck(self.pool(e4))
        d4 = self.up4(bn); d4 = self.dec4(torch.cat([d4, e4], 1) if self.use_skip else d4)
        d3 = self.up3(d4); d3 = self.dec3(torch.cat([d3, e3], 1) if self.use_skip else d3)
        d2 = self.up2(d3); d2 = self.dec2(torch.cat([d2, e2], 1) if self.use_skip else d2)
        d1 = self.up1(d2); d1 = self.dec1(torch.cat([d1, e1], 1) if self.use_skip else d1)
        return self.head(d1)

m = UNet().to(device)
print(f"params {sum(p.numel() for p in m.parameters()):,}")
print("out:", tuple(m(xb[:2].to(device)).shape))

## 3. Speed test

Run this before committing to a training budget. Two details make the difference
between a real measurement and a meaningless one:

- **Warm-up iterations.** The first MPS calls compile Metal kernels and are not
  representative.
- **`sync()` around the timed loop.** Without it you measure queueing time, not
  compute time, and get an absurdly small number.

In [ ]:
m = UNet().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
lf = nn.CrossEntropyLoss()
xb_d, yb_d = xb.to(device), yb.to(device)

for _ in range(3):                                  # warm up: kernel compilation
    loss = lf(m(xb_d), yb_d)
    opt.zero_grad(); loss.backward(); opt.step()
sync()

t0 = time.time()
N = 10
for _ in range(N):
    loss = lf(m(xb_d), yb_d)
    opt.zero_grad(); loss.backward(); opt.step()
sync()
per_step = (time.time() - t0) / N

steps = len(train_dl)
print(f"{per_step*1000:7.0f} ms/step   {steps} steps/epoch")
print(f"{per_step*steps/60:7.1f} min/epoch (training only)")
print(f"{per_step*steps*8*4/60:7.1f} min for 4 arms x 8 epochs (training only)")

    607 ms/step   467 steps/epoch
    4.7 min/epoch (training only)
  151.2 min for 4 arms x 8 epochs (training only)


**Measured on this machine (M5 MacBook Air, MPS):** 607 ms/step, 4.7 min/epoch over
the full 7,470-tile training set — roughly 1.6× slower than the T4 used in Day 8, which
is respectable for a fanless laptop.

Four arms at that rate would be ~2.5 h, so the training set is capped at 2,500 tiles in
section 1. `SIZE` is deliberately left at 256: building and road are small, thin
targets, and halving resolution would damage exactly the classes this notebook is about.

**Thermal note:** a fanless MacBook Air throttles under sustained load. Run on mains
power (battery mode caps GPU wattage) and on a hard surface. Interrupting mid-run only
costs the arm in progress — the loop is sequential.

## 4. The three losses

### Weighted cross-entropy

Re-weights each pixel by its class. The obvious choice, inverse frequency
`w = 1/f`, gives building **435×** the weight of woodland — a single building pixel
contributing as much gradient as 265 woodland pixels. That destabilizes training and
collapses precision as the model over-predicts the rare class.

Inverse *square-root* frequency compresses that to ~16×. It is an empirical compromise
with no principled justification, but it is what works.

### Soft Dice

`Dice = 2|X ∩ Y| / (|X| + |Y|)`, computed **per class and then averaged over classes**.
That is the key structural difference: building contributes as much to the loss as
woodland regardless of pixel count. Weighted CE adjusts pixel weights inside a
per-pixel loss; Dice changes what the loss is computed over.

Softmax probabilities are used instead of `argmax` to keep it differentiable.

### CE + Dice

CE has well-defined per-pixel gradients and gives stable direction early, when Dice's
batch-statistic gradients are noisy. Dice aligns with the evaluation metric. Summing
them is standard practice.

In [ ]:
# Class frequencies measured in Day 8 (300 training tiles, 78.6M pixels)
FREQ = torch.tensor([0.2961, 0.0023, 0.6089, 0.0845, 0.0081])

def sqrt_inv_freq_weights(freq, normalize=True):
    w = 1.0 / freq.sqrt()
    return w / w.mean() if normalize else w

W = sqrt_inv_freq_weights(FREQ).to(device)
for n, f, w in zip(CLASS_NAMES, FREQ.tolist(), W.tolist()):
    print(f"  {n:11s} freq {f*100:6.2f}%   weight {w:6.3f}")
print(f"\n  building/woodland weight ratio: {(W[1]/W[2]).item():.1f}x "
      f"(plain 1/f would be {(FREQ[2]/FREQ[1]).item():.0f}x)")


class SoftDiceLoss(nn.Module):
    """Per-class soft Dice, averaged over classes.

    The smoothing term is added to BOTH numerator and denominator so that a class
    absent from the batch scores Dice = 1 (no penalty) rather than 0. Smoothing the
    denominator only penalizes the model for something it cannot control, and makes
    training mysteriously unstable.
    """
    def __init__(self, n_classes=N_CLASSES, smooth=1.0):
        super().__init__()
        self.n, self.smooth = n_classes, smooth

    def forward(self, logits, target):
        p = F.softmax(logits, dim=1)                       # [N,C,H,W]
        t = F.one_hot(target, self.n).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)      # sum over batch and space, KEEP the class dimension
        inter = (p * t).sum(dims)
        denom = p.sum(dims) + t.sum(dims)
        dice = (2 * inter + self.smooth) / (denom + self.smooth)
        return 1.0 - dice.mean()


class CEPlusDice(nn.Module):
    def __init__(self, weight=None, n_classes=N_CLASSES, alpha=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight)
        self.dice = SoftDiceLoss(n_classes)
        self.alpha = alpha
    def forward(self, logits, target):
        return self.ce(logits, target) + self.alpha * self.dice(logits, target)

  background  freq  29.61%   weight  0.239
  building    freq   0.23%   weight  2.706
  woodland    freq  60.89%   weight  0.166
  water       freq   8.45%   weight  0.447
  road        freq   0.81%   weight  1.442

  building/woodland weight ratio: 16.3x (plain 1/f would be 265x)


### Unit tests

A custom loss needs testing. Dice has several places to get it wrong — the summation
dims, the one-hot channel order, where the smoothing goes — and **none of them raise an
error**. They just train badly, and the conclusion becomes "Dice doesn't help here".

Test 3 is the important one: it checks that a class absent from the batch is not
penalized. Smoothing the denominator only would give those classes Dice = 0, pushing
the mean loss toward 0.8 for reasons the model cannot act on.

In [ ]:
yb_d = yb.to(device)
rand_logits = torch.randn(xb.size(0), N_CLASSES, xb.size(2), xb.size(3), device=device)

print("loss on random logits:")
for name, fn in [("ce", nn.CrossEntropyLoss()),
                 ("wce", nn.CrossEntropyLoss(weight=W)),
                 ("dice", SoftDiceLoss()),
                 ("ce+dice", CEPlusDice(weight=W))]:
    print(f"  {name:8s} {fn(rand_logits, yb_d).item():.4f}")

# 1. near-perfect prediction -> Dice loss ~ 0
perfect = F.one_hot(yb_d, N_CLASSES).permute(0,3,1,2).float() * 20 - 10
print(f"\n  dice(perfect)        {SoftDiceLoss()(perfect, yb_d).item():.6f}   expect ~0")

# 2. confidently wrong everywhere -> Dice loss ~ 1
wrong = F.one_hot((yb_d + 1) % N_CLASSES, N_CLASSES).permute(0,3,1,2).float() * 20 - 10
print(f"  dice(all wrong)      {SoftDiceLoss()(wrong, yb_d).item():.6f}   expect ~1")

# 3. classes absent from the batch must not be penalized
y_single = torch.full_like(yb_d, 2)                        # all woodland
p_single = F.one_hot(y_single, N_CLASSES).permute(0,3,1,2).float() * 20 - 10
print(f"  dice(absent classes) {SoftDiceLoss()(p_single, y_single).item():.6f}   expect ~0")

loss on random logits:
  ce       1.9754
  wce      1.9739
  dice     0.8459
  ce+dice  2.8198

  dice(perfect)        0.000000   expect ~0
  dice(all wrong)      0.999997   expect ~1
  dice(absent classes) 0.001725   expect ~0


## 5. Evaluation and training

`evaluate_v2` adds **precision and recall** alongside IoU. IoU alone cannot separate
"missed the class" from "over-predicted the class", and re-weighting is expected to
trade one for the other — without both numbers there is no way to check whether the
mechanism behaved as intended.

IoU is computed from a confusion matrix accumulated over the whole loader, not averaged
per batch: with building at 0.23% of pixels, many batches contain none, and their
building IoU would be a meaningless 0.

In [ ]:
@torch.no_grad()
def evaluate_v2(model, dl, n_classes=N_CLASSES):
    model.eval()
    cm = torch.zeros(n_classes, n_classes, dtype=torch.int64, device=device)
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(1)
        k = (yb * n_classes + pred).flatten()
        cm += torch.bincount(k, minlength=n_classes**2).reshape(n_classes, n_classes)

    cm = cm.float()
    tp = cm.diag()
    union  = cm.sum(0) + cm.sum(1) - tp
    iou    = tp / union.clamp(min=1)
    prec   = tp / cm.sum(0).clamp(min=1)      # column = predicted
    recall = tp / cm.sum(1).clamp(min=1)      # row    = ground truth
    present = cm.sum(1) > 0
    return dict(pixel_acc=(tp.sum()/cm.sum()).item(),
                miou=iou[present].mean().item(),
                iou=iou.cpu().numpy(), prec=prec.cpu().numpy(),
                recall=recall.cpu().numpy(), cm=cm.cpu().numpy())


def train_seg_v2(model, loss_fn, epochs=6, lr=1e-3, tag="run", verbose=True):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best = {"miou": -1}; hist = []

    for ep in range(epochs):
        t0 = time.time(); model.train(); running = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * xb.size(0)

        m = evaluate_v2(model, val_dl); hist.append(m)
        if verbose:
            per = "  ".join(f"{n[:5]} {v:.3f}" for n, v in zip(CLASS_NAMES, m["iou"]))
            print(f"[{tag}] ep {ep+1}  loss {running/len(train_ds):.4f}  "
                  f"pixAcc {m['pixel_acc']:.4f}  mIoU {m['miou']:.4f}  ({time.time()-t0:.0f}s)")
            print(f"         {per}")

        if m["miou"] > best["miou"]:
            best = m
            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})

    model.load_state_dict(best_state)
    return model, hist, best

## 6. Four arms

`ce` is re-run rather than reusing Day 8's numbers. Day 8 used `train_seg`; this
notebook uses `train_seg_v2`, which records different quantities. **All arms of a
controlled comparison must go through the same code path** — comparing across two
implementations reintroduces exactly the confound the controlled design exists to
remove. It costs one extra arm's worth of compute.

In [ ]:
LOSSES = {
    "ce":      nn.CrossEntropyLoss(),
    "wce":     nn.CrossEntropyLoss(weight=W),
    "dice":    SoftDiceLoss(),
    "ce+dice": CEPlusDice(weight=W),
}

results9, models9 = {}, {}
for name, fn in LOSSES.items():
    seed_all(42)                                   # identical init for every arm
    # 8 epochs, not 6: Dice typically converges more slowly than CE, so a tight
    # epoch budget would penalize it for the wrong reason.
    m, h, best = train_seg_v2(UNet(use_skip=True), fn, epochs=8, tag=name)
    results9[name], models9[name] = best, m
    print()

In [ ]:
import pandas as pd

rows = []
for name, b in results9.items():
    r = {"loss": name, "pixAcc": round(b["pixel_acc"], 4), "mIoU": round(b["miou"], 4)}
    for i, c in enumerate(CLASS_NAMES):
        r[f"IoU_{c[:5]}"] = round(float(b["iou"][i]), 4)
    rows.append(r)
print(pd.DataFrame(rows).set_index("loss"))

print("\nrare classes: precision / recall / IoU")
for ci, cname in [(1, "building"), (4, "road")]:
    print(f"\n  {cname}")
    for name, b in results9.items():
        print(f"    {name:16s} P {b['prec'][ci]:.3f}  R {b['recall'][ci]:.3f}  "
              f"IoU {b['iou'][ci]:.3f}")

               pixAcc    mIoU  IoU_backg  IoU_build  IoU_woodl  IoU_water  IoU_road
loss
ce             0.8677  0.4423     0.8098     0.0000     0.7883     0.5997    0.0136
wce            0.8229  0.4978     0.7332     0.2509     0.7399     0.5777    0.1873
dice           0.8317  0.4987     0.7568     0.2283     0.7375     0.5447    0.2265
ce+dice        0.8014  0.4572     0.7114     0.1857     0.7147     0.5659    0.1081

rare classes: precision / recall / IoU

  building
    ce               P 0.000  R 0.000  IoU 0.000
    wce              P 0.387  R 0.416  IoU 0.251
    dice             P 0.298  R 0.495  IoU 0.228
    ce+dice          P 0.202  R 0.692  IoU 0.186

  road
    ce               P 0.174  R 0.015  IoU 0.014
    wce              P 0.237  R 0.471  IoU 0.187
    dice             P 0.438  R 0.319  IoU 0.226
    ce+dice          P 0.140  R 0.324  IoU 0.108


### Predictions, recorded before the run

| arm | predicted mIoU | actual | predicted building IoU | actual |
|---|---|---|---|---|
| ce | 0.55–0.58 | **0.4423** | 0.30–0.40 | **0.0000** |
| wce | 0.56–0.62 | 0.4978 | 0.35–0.50 | 0.2509 |
| dice | 0.55–0.65 | 0.4987 | 0.35–0.50 | 0.2283 |
| ce+dice | 0.60–0.66 (predicted winner) | **0.4572** | 0.40–0.55 | 0.1857 |

**Every arm fell below the predicted mIoU range, and the predicted winner came third.**

Part of that is explained: the training set was cut from 7,470 to 2,500 tiles after the
predictions were written, and the ranges were not adjusted. Changing the experiment
without re-calibrating the prediction destroys its value as a check.

The `ce+dice` result is not explained by data volume. That was a straightforward
mis-prediction, diagnosed in the next section.

### Reading the results

**Cross-entropy at 2,500 tiles gives building IoU 0.0000** — Day 8's 0.353 at 7,470
tiles has collapsed entirely. Fewer building pixels pushed a class CE was already
neglecting over the edge. Road tells the same story from the other side: precision
0.174 with recall 0.015 means it occasionally guesses right, but misses 98.5% of road
pixels.

**Weighted CE and Dice are statistically indistinguishable** — mIoU 0.4978 vs 0.4987,
a gap of 0.0009. But they reach that tie by opposite routes:

| | building P | building R | road P | road R |
|---|---|---|---|---|
| wce | 0.387 | 0.416 | 0.237 | **0.471** |
| dice | 0.298 | **0.495** | **0.438** | 0.319 |

On building, Dice has the higher recall and weighted CE the higher precision. **On road
the relationship inverts.** A plausible mechanism: weighting is set by frequency, and
road (0.81%) carries more total weighted gradient than building (0.23%) because there
are simply more road pixels, so the model predicts road aggressively — recall up,
precision down. Dice weights every class equally regardless of size and penalizes
overall overlap, so on a thin structure it predicts only where confident — precision up,
recall down.

**None of this is visible from IoU alone.** Logging precision and recall alongside it
is what made the next section possible.

## 7. A design flaw, and the arm that fixes it

The `ce+dice` arm produced building **precision 0.202 against recall 0.692** — it finds
69% of real buildings, but 80% of what it calls a building is not one. That is the
signature of over-prediction, and it points at the implementation:

```python
class CEPlusDice(nn.Module):
    def __init__(self, weight=None, ...):
        self.ce = nn.CrossEntropyLoss(weight=weight)   # instantiated with weight=W
```

`CEPlusDice(weight=W)` uses **weighted** cross-entropy plus Dice. Both terms push toward
predicting rare classes more often. This arm therefore did not test "CE stabilizes,
Dice aligns with the metric" — it tested a double correction, and the correction
overshot.

The fix isolates the combination itself.

In [ ]:
class CEPlusDiceUnweighted(nn.Module):
    """CE + Dice, with UNWEIGHTED cross-entropy.

    The ce+dice arm above used nn.CrossEntropyLoss(weight=W), so both terms push
    toward over-predicting rare classes. This arm lets Dice alone handle class
    balance while CE does what it is there for: stable per-pixel gradients.
    """
    def __init__(self, n_classes=N_CLASSES, alpha=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()          # no class weights
        self.dice = SoftDiceLoss(n_classes)
        self.alpha = alpha
    def forward(self, logits, target):
        return self.ce(logits, target) + self.alpha * self.dice(logits, target)


seed_all(42)
m5, h5, best5 = train_seg_v2(UNet(use_skip=True), CEPlusDiceUnweighted(),
                             epochs=8, tag="ce(plain)+dice")
results9["ce+dice(plain)"] = best5
models9["ce+dice(plain)"] = m5

[ce(plain)+dice] ep 1  loss 1.4953  pixAcc 0.7410  mIoU 0.3179  (186s)
         backg 0.660  build 0.000  woodl 0.552  water 0.378  road 0.000
[ce(plain)+dice] ep 2  loss 1.1905  pixAcc 0.6962  mIoU 0.2943  (191s)
         backg 0.637  build 0.000  woodl 0.522  water 0.295  road 0.018
[ce(plain)+dice] ep 3  loss 1.0935  pixAcc 0.8090  mIoU 0.4325  (233s)
         backg 0.715  build 0.000  woodl 0.688  water 0.618  road 0.142
[ce(plain)+dice] ep 4  loss 1.0151  pixAcc 0.8469  mIoU 0.4702  (239s)
         backg 0.768  build 0.039  woodl 0.762  water 0.622  road 0.160
[ce(plain)+dice] ep 5  loss 0.9707  pixAcc 0.8174  mIoU 0.4515  (227s)
         backg 0.759  build 0.121  woodl 0.692  water 0.496  road 0.188
[ce(plain)+dice] ep 6  loss 0.9365  pixAcc 0.8418  mIoU 0.5063  (226s)
         backg 0.755  build 0.189  woodl 0.742  water 0.649  road 0.197
[ce(plain)+dice] ep 7  loss 0.8589  pixAcc 0.8260  mIoU 0.5042  (223s)
         backg 0.764  build 0.293  woodl 0.705  water 0.544  road 0.216

In [ ]:
rows = []
for name, b in results9.items():
    r = {"loss": name, "pixAcc": round(b["pixel_acc"], 4), "mIoU": round(b["miou"], 4)}
    for i, c in enumerate(CLASS_NAMES):
        r[f"IoU_{c[:5]}"] = round(float(b["iou"][i]), 4)
    rows.append(r)
print(pd.DataFrame(rows).set_index("loss"))

print("\nrare classes: precision / recall / IoU")
for ci, cname in [(1, "building"), (4, "road")]:
    print(f"\n  {cname}")
    for name, b in results9.items():
        print(f"    {name:16s} P {b['prec'][ci]:.3f}  R {b['recall'][ci]:.3f}  "
              f"IoU {b['iou'][ci]:.3f}")

                pixAcc    mIoU  IoU_backg  IoU_build  IoU_woodl  IoU_water  IoU_road
loss
ce              0.8677  0.4423     0.8098     0.0000     0.7883     0.5997    0.0136
wce             0.8229  0.4978     0.7332     0.2509     0.7399     0.5777    0.1873
dice            0.8317  0.4987     0.7568     0.2283     0.7375     0.5447    0.2265
ce+dice         0.8014  0.4572     0.7114     0.1857     0.7147     0.5659    0.1081
ce+dice(plain)  0.8566  0.5192     0.7888     0.2257     0.7652     0.6149    0.2014

rare classes: precision / recall / IoU

  building
    ce               P 0.000  R 0.000  IoU 0.000
    wce              P 0.387  R 0.416  IoU 0.251
    dice             P 0.298  R 0.495  IoU 0.228
    ce+dice          P 0.202  R 0.692  IoU 0.186
    ce+dice(plain)   P 0.342  R 0.399  IoU 0.226

  road
    ce               P 0.174  R 0.015  IoU 0.014
    wce              P 0.237  R 0.471  IoU 0.187
    dice             P 0.438  R 0.319  IoU 0.226
    ce+dice          P 0.140  R 0

### The diagnosis holds

Changing one thing — removing class weights from the CE term:

| | building P | building R | road P | road R | mIoU |
|---|---|---|---|---|---|
| ce+dice (weighted CE) | 0.202 | 0.692 | 0.140 | 0.324 | 0.4572 |
| **ce+dice (plain CE)** | **0.342** | 0.399 | **0.458** | 0.264 | **0.5192** |

**Precision rises sharply, recall falls, IoU rises net.** Building precision +69%, road
precision +227%. With Dice alone handling class balance and CE providing stable
per-pixel gradients, the combination does what it was supposed to do.

### Final ranking

```
ce+dice(plain)  0.5192
dice            0.4987
wce             0.4978
ce+dice         0.4572
ce              0.4423
```

Judged against the run-to-run variance measured in Day 6 (~1.2 points on a comparable
setup):

- **Winner vs `ce`: 7.7 points.** Well outside noise. Established.
- **Winner vs second: 2.05 points.** Above the measured variance but not by a wide
  margin. Suggestive, not established — no multi-seed study was run.
- **`dice` vs `wce`: 0.0009 points.** Indistinguishable. They are tied.

### The finding that matters most

```
ce              pixAcc 0.8677   mIoU 0.4423   building IoU 0.0000
ce+dice(plain)  pixAcc 0.8566   mIoU 0.5192   building IoU 0.2257
```

**Selecting a model by pixel accuracy picks the one arm that cannot see buildings at
all.** The accuracy difference is 1.1 points; the mIoU difference is 7.7, and building
goes from nothing to something.

This is the third time in three notebooks that an aggregate metric has concealed a
per-class failure — Day 6 on CIFAR-10 (20-point F1 spread), Day 8 on the skip ablation
(0.6% accuracy hiding two lost classes), and now in model selection itself.

## Limitations

- **Single seed per arm.** Day 6 measured run-to-run variation of 1.22 points on a
  comparable setup. The 2.05-point lead of `ce+dice(plain)` over `dice` is above that
  but not decisively — the ranking of the top three is not established. The gap to `ce`
  (7.7 points) and the categorical failures are unaffected.
- **Trained on 2,500 of 7,470 tiles** for compute reasons (4.7 min/epoch on this
  machine). All arms are affected identically so the comparison stands, but absolute
  IoU is lower than Day 8's and the two notebooks must not be compared directly — CE's
  building IoU fell from 0.353 to 0.000 on this reduction alone.
- **Eight epochs, no early stopping.** `ce+dice(plain)` was still improving at epoch 8
  (mIoU 0.5042 → 0.5192). All numbers are floors.
- **Per-class IoU is noisy across epochs.** `ce+dice(plain)` building peaked at 0.293
  (epoch 7) and the selected checkpoint reports 0.226 (epoch 8). The best-mIoU
  checkpoint is not the best-building checkpoint, and single-class figures should not
  be over-read.
- **`alpha=1.0` in both combined losses is untuned.** On random logits CE was 1.97 and
  Dice 0.85, so CE dominates the gradient at this setting. The balance is a
  hyperparameter that was never swept.
- **Weights derived from a 300-tile sample** of the training set, not the full split.
- **Thermal throttling.** Epoch times rose from 186 s to 232 s within a single arm on
  this fanless machine. It affects wall-clock, not results, but means later arms ran
  under slightly different conditions than earlier ones.

## Summary

| loss | pixAcc | mIoU | building IoU | road IoU |
|---|---|---|---|---|
| ce | **0.8677** | 0.4423 | **0.0000** | 0.0136 |
| wce | 0.8229 | 0.4978 | 0.2509 | 0.1873 |
| dice | 0.8317 | 0.4987 | 0.2283 | 0.2265 |
| ce+dice (weighted CE) | 0.8014 | 0.4572 | 0.1857 | 0.1081 |
| **ce+dice (plain CE)** | 0.8566 | **0.5192** | 0.2257 | 0.2014 |

1. **Cross-entropy abandons a 0.23% class entirely** — building IoU exactly 0.0000, road
   recall 0.015.
2. **Weighted CE and Dice tie on IoU while making opposite precision/recall
   trade-offs**, and invert that relationship between building and road. IoU alone
   cannot show this.
3. **Combining weighted CE with Dice double-corrects and overshoots.** Using plain CE
   instead raised building precision 69% and produced the best mIoU.
4. **Pixel accuracy selects the worst model.** The highest-accuracy arm is the only one
   with zero building IoU.
5. **The design flaw in (3) was only visible because precision and recall were logged.**
   The instrumentation decision, not the loss function, is what produced the best result
   here.

**Next:** real geospatial data pipelines — tiling GeoTIFFs, per-band normalization,
nodata handling — and then blocked spatial validation, where random train/val splits
leak because neighbouring patches are not independent.